# RL Lab Exam — One Template, Thirteen Programs

Every program has the **same four blocks in the same order**. Only the `POLICY`, `ENVIRONMENT` and `EQUATION` lines change.
Same names everywhere: `state, action, next_state, reward, alpha, gamma, epsilon, Q, V`.
Run the cells top to bottom (Google Colab ready).

## The Template (memorize this once)

```python
# 1. SETUP                      -> alpha, gamma, epsilon, episodes
# 2. STATES AND ACTIONS         -> states, goal, moves (actions), Q or V table (or networks)
# 3. TRAINING LOOP
for episode in range(episodes):
    state = start
    while state != goal:              # (or: for step in range(...))
        # POLICY                      -> how to choose action
        # ENVIRONMENT                 -> next_state, reward
        # EQUATION                    -> the learning update   <-- changes per algorithm
        state = next_state
# 4. OUTPUT                     -> print the table / reward
```

**What goes in each slot**

| Slot | Tabular (1-6, 10, 12, 13) | Neural (7, 8, 9, 11) |
|------|------------------|----------------------|
| POLICY | random / epsilon-greedy / always right | `Categorical(actor(x)).sample()` or epsilon-greedy on `model(x)` |
| ENVIRONMENT | `next_state`, `reward` | `next_state`, `reward` |
| EQUATION | one update line (below) | a `loss`, then `zero_grad / backward / step` |

**The equations (the only thing that really changes)**
```python
# TD(0)         V[state] += alpha * (reward + gamma * V[next_state] - V[state])      (also Tic-Tac-Toe)
# Bandit        Q[action] += alpha * (reward - Q[action])                              (no next state)
# Q-Learning    Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])
# SARSA         Q[state, action] += alpha * (reward + gamma * Q[next_state, next_action] - Q[state, action])
# Dyna-Q        Q-Learning update, then store model[(state, action)] = (reward, next_state), then n replayed updates
# Ind. Q-Learn  Q-Learning update on Q[agent]
# REINFORCE     loss = -(log_prob * G)                                   # G = discounted return
# Actor-Critic  advantage = reward + gamma * V(next) - V(state)
#               loss = -log_prob * advantage + advantage ** 2
# DQN           loss = (Q(state, action) - (reward + gamma * max Q(next_state))) ** 2
# PPO           ratio = exp(new_log_prob - old_log_prob)
#               loss = -min(ratio * advantage, clip(ratio, 0.8, 1.2) * advantage) + advantage ** 2
```

**Shared helpers**
```python
# Epsilon-greedy (tabular)                  # Discounted returns (REINFORCE, PPO)
if random.random() < epsilon:               G = 0
    action = random.randint(0, n_actions-1) returns = []
else:                                       for reward in reversed(rewards):
    action = np.argmax(Q[state])                G = reward + gamma * G
                                                returns.insert(0, G)

# Neural nets                                # Neural update ending
actor  = nn.Sequential(nn.Linear(n_in,16), nn.ReLU(), nn.Linear(16,n_actions), nn.Softmax(dim=-1))
critic = nn.Sequential(nn.Linear(n_in,16), nn.ReLU(), nn.Linear(16,1))
optimizer.zero_grad(); loss.backward(); optimizer.step()
```

Line world (programs 1-5, 10): `0 - 1 - 2 - 3 - 4(Goal)`, reward `10` at goal, `-1` otherwise.

## 1. TD(0) Grid World — Random Selection with Restricted States
- Policy = random action. `V[state]` is learned with the TD(0) equation.
- State `2` is **restricted**: moving into it is blocked (agent stays).
- Fix vs. original: a blocked state on a plain left/right line can never be passed (infinite loop), so actions are `-1, +1, +2` and the agent can **jump over** state 2.
- Expected output: `V` rises toward the goal (V[3] about 10, V[4] = 0, V[2] = 0 never visited).

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
episodes = 100

# 2. STATES AND ACTIONS
n_states = 5
goal = 4
restricted = 2
moves = [-1, 1, 2]                                      # left, right, jump
n_actions = len(moves)
V = np.zeros(n_states)

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0
    while state != goal:
        # POLICY: random
        action = random.randint(0, n_actions - 1)

        # ENVIRONMENT
        next_state = max(0, min(goal, state + moves[action]))
        if next_state == restricted:
            next_state = state
        reward = 10 if next_state == goal else -1

        # EQUATION: TD(0)
        V[state] += alpha * (
            reward + gamma * V[next_state] - V[state]
        )

        state = next_state

# 4. OUTPUT
print("V:", np.round(V, 2))

## 2. TD(0) Grid World — Epsilon-Greedy with Restricted States
- Same as program 1. **Only the POLICY block changes.**
- Epsilon-greedy: with probability `epsilon` explore randomly, otherwise take the greedy move (right).
- Exploration is what lets the agent get past the blocked state 2.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 100

# 2. STATES AND ACTIONS
n_states = 5
goal = 4
restricted = 2
moves = [-1, 1, 2]                                      # left, right, jump
n_actions = len(moves)
V = np.zeros(n_states)

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0
    while state != goal:
        # POLICY: epsilon-greedy            <-- only this block differs from program 1
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = 1                                  # greedy = right

        # ENVIRONMENT
        next_state = max(0, min(goal, state + moves[action]))
        if next_state == restricted:
            next_state = state
        reward = 10 if next_state == goal else -1

        # EQUATION: TD(0)
        V[state] += alpha * (
            reward + gamma * V[next_state] - V[state]
        )

        state = next_state

# 4. OUTPUT
print("V:", np.round(V, 2))

## 3. TD(0) Grid World — Correct Policy vs Epsilon-Greedy
- Same template twice, with two value tables and the same TD(0) equation.
- **Correct policy**: always move right (no restriction) -> clean values.
- **Epsilon-greedy (restricted)**: explores and wastes steps -> lower values.
- Expected output: `V_correct` is higher than `V_eps` (state 2 stays 0 for `V_eps`: blocked, never visited).

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 100

# 2. STATES AND ACTIONS
n_states = 5
goal = 4
restricted = 2
moves = [-1, 1, 2]                                      # left, right, jump
n_actions = len(moves)
V_correct = np.zeros(n_states)
V_eps = np.zeros(n_states)

# 3a. TRAINING LOOP: correct policy
for episode in range(episodes):
    state = 0
    while state != goal:
        # POLICY: always right
        action = 1

        # ENVIRONMENT (no restriction)
        next_state = max(0, min(goal, state + moves[action]))
        reward = 10 if next_state == goal else -1

        # EQUATION: TD(0)
        V_correct[state] += alpha * (
            reward + gamma * V_correct[next_state] - V_correct[state]
        )

        state = next_state

# 3b. TRAINING LOOP: epsilon-greedy with restricted state
for episode in range(episodes):
    state = 0
    while state != goal:
        # POLICY: epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = 1

        # ENVIRONMENT (state 2 blocked)
        next_state = max(0, min(goal, state + moves[action]))
        if next_state == restricted:
            next_state = state
        reward = 10 if next_state == goal else -1

        # EQUATION: TD(0)
        V_eps[state] += alpha * (
            reward + gamma * V_eps[next_state] - V_eps[state]
        )

        state = next_state

# 4. OUTPUT
print("State | Correct | Eps-Greedy")
for s in range(n_states):
    print(s, "    |", round(V_correct[s], 2), "  |", round(V_eps[s], 2))

## 4. Q-Learning Grid World
- `Q[state, action]` table: 5 states x 2 actions (0 = left, 1 = right).
- POLICY is epsilon-greedy on `Q`. EQUATION uses `max` over the next state (off-policy).
- Fix vs. original: Q-table was `(5, 4)` for 2 actions -> now `(5, 2)`.
- Expected output: learned path `1, 2, 3, 4`.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

# 2. STATES AND ACTIONS
n_states = 5
goal = 4
moves = [-1, 1]                                         # action 0 = left, 1 = right
n_actions = len(moves)
Q = np.zeros((n_states, n_actions))

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0
    while state != goal:
        # POLICY: epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = np.argmax(Q[state])

        # ENVIRONMENT
        next_state = max(0, min(goal, state + moves[action]))
        reward = 10 if next_state == goal else -1

        # EQUATION: Q-Learning
        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state])
            - Q[state, action]
        )

        state = next_state

# 4. OUTPUT
print("Q-table:\n", np.round(Q, 2))

state = 0
path = []
for step in range(10):                                  # greedy test run
    state = max(0, min(goal, state + moves[np.argmax(Q[state])]))
    path.append(state)
    if state == goal:
        break
print("Path:", path)

Q-table:
 [[ 3.11  4.58]
 [ 3.11  6.2 ]
 [ 4.56  8.  ]
 [ 6.19 10.  ]
 [ 0.    0.  ]]
Path: [1, 2, 3, 4]


## 5. SARSA (Grid World)
- Same as program 4 (Q-Learning): same line world, same `Q`, same epsilon-greedy policy. **Three changes**, marked `# SARSA`.
- SARSA is **on-policy**: it picks `next_action` *before* the update and uses `Q[next_state, next_action]` (the action it will really take). Q-Learning uses `np.max(Q[next_state])` (the best action).
- Name = **S**tate, **A**ction, **R**eward, next **S**tate, next **A**ction.
- Expected output: learned path `1, 2, 3, 4`.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

# 2. STATES AND ACTIONS
n_states = 5
goal = 4
moves = [-1, 1]                                         # action 0 = left, 1 = right
n_actions = len(moves)
Q = np.zeros((n_states, n_actions))

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0

    # POLICY: first action                              # SARSA 1: choose action before the loop
    if random.random() < epsilon:
        action = random.randint(0, n_actions - 1)
    else:
        action = np.argmax(Q[state])

    while state != goal:
        # ENVIRONMENT
        next_state = max(0, min(goal, state + moves[action]))
        reward = 10 if next_state == goal else -1

        # POLICY: next action                           # SARSA 2: choose next_action now
        if random.random() < epsilon:
            next_action = random.randint(0, n_actions - 1)
        else:
            next_action = np.argmax(Q[next_state])

        # EQUATION: SARSA                               # SARSA 3: Q[next_state, next_action]
        Q[state, action] += alpha * (
            reward + gamma * Q[next_state, next_action]
            - Q[state, action]
        )

        state = next_state
        action = next_action

# 4. OUTPUT
print("Q-table:\n", np.round(Q, 2))

state = 0
path = []
for step in range(10):                                  # greedy test run
    state = max(0, min(goal, state + moves[np.argmax(Q[state])]))
    path.append(state)
    if state == goal:
        break
print("Path:", path)

## 6. Dyna-Q for Traffic Signal Control
- State = (NS queue, EW queue), each capped at 4. Action 0 = NS green, 1 = EW green. Reward = `-(ns + ew)`.
- EQUATION = Q-Learning update, then **store** `model[(state, action)] = (reward, next_state)`, then **plan**: replay `n` random stored experiences with the same update.
- Fix vs. original: added epsilon-greedy exploration and several planning steps.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 600
totals = []

# 2. STATES AND ACTIONS
n_actions = 2                  # 0 = NS green, 1 = EW green
Q = np.zeros((5, 5, n_actions))
model = {}
n = 5                          # planning steps

# 3. TRAINING LOOP
for episode in range(episodes):
    ns, ew = 2, 2
    rewards = []

    for step in range(20):
        state = (min(ns, 4), min(ew, 4))

        # POLICY: epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = np.argmax(Q[state])

        # ENVIRONMENT
        ns += random.randint(0, 1)
        ew += random.randint(0, 1)

        if action == 0:
            ns = max(0, ns - 2)
        else:
            ew = max(0, ew - 2)

        next_state = (min(ns, 4), min(ew, 4))
        reward = -(ns + ew)
        rewards.append(reward)

        # EQUATION: Q-Learning
        Q[state][action] += alpha * (
            reward + gamma * np.max(Q[next_state])
            - Q[state][action]
        )

        # EQUATION: Dyna-Q = store + plan
        model[(state, action)] = (reward, next_state)

        for _ in range(n):
            (s, a), (r, s2) = random.choice(list(model.items()))

            Q[s][a] += alpha * (
                r + gamma * np.max(Q[s2]) - Q[s][a]
            )

    totals.append(sum(rewards))

    print(
        "Episode", episode + 1,
        "Final NS queue:", ns,
        "Final EW queue:", ew,
        "Total queue:", ns + ew
    )

# 4. OUTPUT
print("Avg reward, first 100 episodes:", sum(totals[:100]) / 100)
print("Avg reward, last 100 episodes: ", sum(totals[-100:]) / 100)

print(
    "Best signal when NS queue=4, EW queue=0:",
    "NS" if np.argmax(Q[4, 0]) == 0 else "EW"
)
print(
    "Best signal when NS queue=0, EW queue=4:",
    "NS" if np.argmax(Q[0, 4]) == 0 else "EW"
)

Episode 1 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 2 Final NS queue: 0 Final EW queue: 1 Total queue: 1
Episode 3 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 4 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 5 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 6 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 7 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 8 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 9 Final NS queue: 3 Final EW queue: 1 Total queue: 4
Episode 10 Final NS queue: 1 Final EW queue: 0 Total queue: 1
Episode 11 Final NS queue: 1 Final EW queue: 0 Total queue: 1
Episode 12 Final NS queue: 0 Final EW queue: 1 Total queue: 1
Episode 13 Final NS queue: 1 Final EW queue: 0 Total queue: 1
Episode 14 Final NS queue: 0 Final EW queue: 1 Total queue: 1
Episode 15 Final NS queue: 1 Final EW queue: 0 Total queue: 1
Episode 16 Final NS queue: 0 Final EW queue: 0 Total queue: 0
Episode 17 Final 

## 7. Monte Carlo Policy Gradient for Traffic Signal Control
- Same traffic world as program 6. POLICY = a network giving probabilities of [NS green, EW green].
- Play a **whole episode**, saving `log_prob` and `reward` of every step. EQUATION runs **after** the episode.
- EQUATION: discounted returns `G`, then `loss = -(log_prob * G)`.
- Expected output: the average reward of the last 100 episodes is less negative than the first 100.

In [ ]:

import torch
import torch.nn as nn
import random

# 1. SETUP
gamma = 0.9
episodes = 600
totals =[]

# 2. STATES AND ACTIONS
n_actions = 2                  # 0 = NS green, 1 = EW green

actor = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, n_actions), nn.Softmax(dim=-1)
)

optimizer = torch.optim.Adam(actor.parameters(), lr=0.01)

# 3. TRAINING LOOP
for episode in range(episodes):
    ns, ew = 5, 5
    log_probs = []
    rewards = []

    for step in range(20):
        x = torch.tensor(
            [ns / 20, ew / 20], dtype=torch.float32
        )

        # POLICY: actor
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()
        log_probs.append(dist.log_prob(action))

        # ENVIRONMENT
        ns += random.randint(0, 3)
        ew +=  random.randint(0, 3)

        if action.item() == 0:
            ns = max(0, ns - 4)
        else:
            ew = max(0, ew - 4)

        reward = -(ns + ew)
        rewards.append(reward)

    # EQUATION: Monte Carlo Policy Gradient (REINFORCE)
    G = 0
    returns = []

    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    loss = -(torch.stack(log_probs) * returns).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    totals.append(sum(rewards))


    print(
        "Episode", episode + 1,
        "Final NS queue:", ns,
        "Final EW queue:", ew,
        "Total queue:", ns + ew
    )

# 4. OUTPUT
print("Avg reward, first 100 episodes:", sum(totals[:100]) / 100)
print("Avg reward, last 100 episodes: ", sum(totals[-100:]) / 100)


Episode 1 Final NS queue: 6 Final EW queue: 2 Total queue: 8
Episode 2 Final NS queue: 13 Final EW queue: 0 Total queue: 13
Episode 3 Final NS queue: 0 Final EW queue: 6 Total queue: 6
Episode 4 Final NS queue: 11 Final EW queue: 0 Total queue: 11
Episode 5 Final NS queue: 2 Final EW queue: 2 Total queue: 4
Episode 6 Final NS queue: 5 Final EW queue: 2 Total queue: 7
Episode 7 Final NS queue: 14 Final EW queue: 0 Total queue: 14
Episode 8 Final NS queue: 4 Final EW queue: 0 Total queue: 4
Episode 9 Final NS queue: 14 Final EW queue: 0 Total queue: 14
Episode 10 Final NS queue: 9 Final EW queue: 0 Total queue: 9
Episode 11 Final NS queue: 28 Final EW queue: 3 Total queue: 31
Episode 12 Final NS queue: 13 Final EW queue: 0 Total queue: 13
Episode 13 Final NS queue: 16 Final EW queue: 0 Total queue: 16
Episode 14 Final NS queue: 19 Final EW queue: 6 Total queue: 25
Episode 15 Final NS queue: 6 Final EW queue: 0 Total queue: 6
Episode 16 Final NS queue: 20 Final EW queue: 2 Total queue: 22

## 8. Actor-Critic for Autonomous Vehicle Speed Control
- State = speed. Actions: brake (-5), maintain (0), accelerate (+5). Target speed = 60. Reward = `-|target - speed| / 10`.
- **Actor** = POLICY network (action probabilities). **Critic** = value network `V(state)`.
- EQUATION (every step): `advantage = reward + gamma * V(next) - V(state)`, `loss = -log_prob * advantage + advantage ** 2`.
- Expected output: final speed gets close to 60.

In [ ]:

import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.9
episodes = 100

# 2. STATES AND ACTIONS
target_speed = 60
moves = [-5, 0, 5]                 # brake, maintain, accelerate
n_actions = len(moves)

actor = nn.Sequential(nn.Linear(1, 16), nn.ReLU(),
                      nn.Linear(16, n_actions), nn.Softmax(dim=-1))
critic = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))

optimizer = torch.optim.Adam(
    list(actor.parameters()) + list(critic.parameters()), lr=0.01
)

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0                       # state = speed

    for step in range(50):
        x = torch.tensor([state / 100], dtype=torch.float32)

        # POLICY: actor
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()

        # ENVIRONMENT
        next_state = max(0, min(100, state + moves[action.item()]))
        reward = -abs(target_speed - next_state) / 10
        next_x = torch.tensor([next_state / 100], dtype=torch.float32)

        # EQUATION: Actor-Critic
        value = critic(x).squeeze()
        next_value = critic(next_x).squeeze().detach()
        target = reward + gamma * next_value
        advantage = target - value

        loss = -dist.log_prob(action) * advantage.detach() + advantage ** 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state
        if abs(state - target_speed) < 2:
            break

    print("Episode", episode + 1, "Final speed:", state)


## 9. Deep Q-Network (DQN) for Autonomous Vehicle Speed Control
- Same speed environment as program 8. A network replaces the Q-table: input speed -> 3 Q-values.
- POLICY: epsilon-greedy on the network (same as the tabular programs, `argmax` of `model(x)`).
- EQUATION: `loss = (Q(state, action) - (reward + gamma * max Q(next_state))) ** 2`. Same idea as Q-Learning (program 4).
- Expected output: final speed gets close to 60.

In [ ]:
import torch
import torch.nn as nn
import random

# 1. SETUP
gamma = 0.9
epsilon = 0.2
episodes = 100

# 2. STATES AND ACTIONS
target_speed = 60
moves = [-5, 0, 5]                                      # brake, maintain, accelerate
n_actions = len(moves)
model = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, n_actions))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0                                           # state = speed
    for step in range(50):
        x = torch.tensor([state / 100], dtype=torch.float32)

        # POLICY: epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = model(x).argmax().item()

        # ENVIRONMENT
        next_state = max(0, min(100, state + moves[action]))
        reward = -abs(target_speed - next_state) / 10
        next_x = torch.tensor([next_state / 100], dtype=torch.float32)

        # EQUATION: DQN
        q_value = model(x)[action]
        target = reward + gamma * model(next_x).max().detach()
        loss = (q_value - target) ** 2
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state
        if abs(state - target_speed) < 2:
            break

    if (episode + 1) % 20 == 0:
        print("Episode", episode + 1, "Final speed:", state)

## 10. Independent Q-Learning
- Two agents, each with its **own** Q-table (`Q[agent]`), learning in its own copy of the line world.
- Identical to program 4. **Only change:** an extra `for agent in range(agents)` loop and an `agent` index on `Q`.
- Expected output: both agents learn path `[0, 1, 2, 3, 4]`.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

# 2. STATES AND ACTIONS
agents = 2
n_states = 5
goal = 4
moves = [-1, 1]                                         # action 0 = left, 1 = right
n_actions = len(moves)
Q = np.zeros((agents, n_states, n_actions))             # one table per agent

# 3. TRAINING LOOP
for episode in range(episodes):
    for agent in range(agents):                         # NEW: loop over agents
        state = 0
        while state != goal:
            # POLICY: epsilon-greedy
            if random.random() < epsilon:
                action = random.randint(0, n_actions - 1)
            else:
                action = np.argmax(Q[agent, state])

            # ENVIRONMENT
            next_state = max(0, min(goal, state + moves[action]))
            reward = 10 if next_state == goal else -1

            # EQUATION: Q-Learning (per agent)
            Q[agent, state, action] += alpha * (
                reward + gamma * np.max(Q[agent, next_state])
                - Q[agent, state, action]
            )

            state = next_state

# 4. OUTPUT
for agent in range(agents):
    state = 0
    path = [state]
    for step in range(10):                              # greedy test run
        state = max(0, min(goal, state + moves[np.argmax(Q[agent, state])]))
        path.append(state)
        if state == goal:
            break
    print("Agent", agent + 1, "path:", path)

## 11. Proximal Policy Optimization (PPO) with an Actor-Critic Network
PPO = **REINFORCE (7) + Actor-Critic (8) networks + 3 changes** (marked `# PPO` in the code):
- `# PPO 1`: save the **old log probs** while playing the episode.
- `# PPO 2`: repeat the EQUATION several times on the same episode, with `ratio = exp(new_log_prob - old_log_prob)`.
- `# PPO 3`: **clip** the ratio to `[0.8, 1.2]` and take the `min`.
- Environment: Gymnasium `CartPole-v1` (the only program that needs Gymnasium).
- Expected output: episode reward rises from about 20 toward 200+.

In [ ]:
!pip install -q gymnasium

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.9
episodes = 200

# 2. STATES AND ACTIONS
env = gym.make("CartPole-v1")                           # state: 4 numbers, 2 actions
n_actions = 2
actor = nn.Sequential(
    nn.Linear(4, 32), nn.ReLU(),
    nn.Linear(32, n_actions), nn.Softmax(dim=-1)
)
critic = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))
optimizer = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=0.003)

# 3. TRAINING LOOP
for episode in range(episodes):
    state, _ = env.reset()
    xs, actions, old_log_probs, rewards = [], [], [], []
    done = False

    while not done:
        x = torch.tensor(state, dtype=torch.float32)

        # POLICY: actor
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()
        xs.append(x)
        actions.append(action)
        old_log_probs.append(dist.log_prob(action).detach())   # PPO 1

        # ENVIRONMENT
        state, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    # EQUATION: PPO (after the episode)
    G = 0                                               # returns (same as REINFORCE)
    returns = []
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    xs = torch.stack(xs)
    actions = torch.stack(actions)
    old_log_probs = torch.stack(old_log_probs)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    for _ in range(4):                                  # PPO 2: several updates
        dist = torch.distributions.Categorical(actor(xs))
        new_log_probs = dist.log_prob(actions)
        value = critic(xs).squeeze(-1)
        advantage = returns - value                     # same idea as Actor-Critic

        ratio = torch.exp(new_log_probs - old_log_probs)
        clipped = torch.clamp(ratio, 0.8, 1.2)          # PPO 3: clip
        actor_loss = -torch.min(ratio * advantage.detach(), clipped * advantage.detach()).mean()
        critic_loss = (advantage ** 2).mean()

        loss = actor_loss + critic_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (episode + 1) % 20 == 0:
        print("Episode", episode + 1, "Reward:", sum(rewards))

# 4. OUTPUT
env.close()

## 12. Basic RL: TD(0) Tic-Tac-Toe with Episode Display
- **Same as program 1**: random policy + the same TD(0) equation. Only the world is different.
- `state` = the board as a 9-character string (`X`, `O`, space). `action` = an empty square (0-8). X and O take turns playing random moves.
- Reward: X wins `+10`, O wins `-10`, otherwise `0`. The game ends on a win or a full board.
- Episode display: every 1000 episodes the final board and result are printed (`10` = X wins, `-10` = O wins, `0` = draw).
- Expected output: small values near 0 for the opening boards (random play is fairly even; X moves first so it leans slightly positive). Values get larger for boards close to a win. This program **evaluates** random play; it does not improve the play.

In [ ]:
import random
from collections import defaultdict

# 1. SETUP
alpha = 0.1
gamma = 0.9
episodes = 3000

# 2. STATES AND ACTIONS
lines = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]
V = defaultdict(float)                                  # V[board], starts at 0

# 3. TRAINING LOOP
for episode in range(episodes):
    state = " " * 9                                     # empty board
    player = "X"
    done = False
    while not done:
        # POLICY: random empty square
        action = random.choice([i for i in range(9) if state[i] == " "])

        # ENVIRONMENT
        next_state = state[:action] + player + state[action + 1:]
        reward = 0
        for a, b, c in lines:                           # three in a row?
            if next_state[a] == next_state[b] == next_state[c] == player:
                reward = 10 if player == "X" else -10
                done = True
        if " " not in next_state:                       # board full
            done = True

        # EQUATION: TD(0)
        V[state] += alpha * (
            reward + gamma * V[next_state] - V[state]
        )

        state = next_state
        player = "O" if player == "X" else "X"

    # EPISODE DISPLAY
    if (episode + 1) % 1000 == 0:
        print("Episode", episode + 1, "Result:", reward)
        print(state[0:3])
        print(state[3:6])
        print(state[6:9])

# 4. OUTPUT
print("V(empty board):", round(V[" " * 9], 3))
print("V(X in centre):", round(V[" " * 4 + "X" + " " * 4], 3))
print("V(X in corner):", round(V["X" + " " * 8], 3))

## 13. Multi-Armed Bandit (Epsilon-Greedy)
- No states, only **actions = arms**. `Q[action]` = estimated reward of each arm. Each pull is one episode.
- POLICY: epsilon-greedy (same lines as program 4). ENVIRONMENT: reward = noisy payout around the arm's true mean.
- EQUATION = Q-Learning **without the `gamma * max Q[next_state]` term** (nothing comes after a pull): `Q[action] += alpha * (reward - Q[action])`.
- Expected output: `Q` is close to `true_means`, and the best arm (index 1) is pulled most.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
epsilon = 0.2
episodes = 1000

# 2. STATES AND ACTIONS
true_means = [2, 5, 3, 1]                               # hidden average reward of each arm
n_actions = len(true_means)
Q = np.zeros(n_actions)
pulls = np.zeros(n_actions)

# 3. TRAINING LOOP
for episode in range(episodes):
    # POLICY: epsilon-greedy
    if random.random() < epsilon:
        action = random.randint(0, n_actions - 1)
    else:
        action = np.argmax(Q)

    # ENVIRONMENT
    reward = random.gauss(true_means[action], 1)
    pulls[action] += 1

    # EQUATION: Bandit (Q-Learning without the next-state term)
    Q[action] += alpha * (
        reward - Q[action]
    )

# 4. OUTPUT
print("Estimated Q:", np.round(Q, 2))
print("True means: ", true_means)
print("Pulls:      ", pulls)
print("Best arm:", np.argmax(Q))

## Quick Revision — Same Template, Different Equation

| # | Algorithm | EQUATION slot | Extra |
|---|-----------|---------------|-------|
| 1 | TD(0) random | `V += a*(r + g*V[next] - V[state])` | policy = random |
| 2 | TD(0) epsilon-greedy | same | policy = epsilon-greedy |
| 3 | TD(0) correct vs eps | same, two tables | two policies |
| 4 | Q-Learning | `Q += a*(r + g*max Q[next] - Q)` | epsilon-greedy |
| 5 | SARSA | `Q += a*(r + g*Q[next_state, next_action] - Q)` | on-policy: choose `next_action` before the update |
| 6 | Dyna-Q | Q-Learning, then store `model`, then `n` replays | planning loop |
| 7 | REINFORCE | `loss = -log_prob * G` after the episode | returns |
| 8 | Actor-Critic | `adv = r + g*V(next) - V`; `-log_prob*adv + adv^2` every step | actor + critic |
| 9 | DQN | `(Q(s,a) - (r + g*max Q(next)))^2` | network instead of table |
| 10 | Independent Q-Learning | Q-Learning per agent | `for agent` loop |
| 11 | PPO | REINFORCE + old log probs + `-min(ratio*adv, clip(ratio)*adv)` | several update epochs |
| 12 | TD(0) Tic-Tac-Toe | same TD(0) on board values `V[board]` | random X and O, episode display |
| 13 | Multi-Armed Bandit | `Q[action] += a*(r - Q[action])` | no states, epsilon-greedy |

- **Table (NumPy / dict):** 1-6, 10, 12, 13. **Network (PyTorch):** 7, 8, 9, 11.
- **Value-based:** TD(0) (incl. Tic-Tac-Toe), Q-Learning, SARSA, Dyna-Q, DQN, IQL, Bandit. **Policy-based:** REINFORCE, Actor-Critic, PPO.
- **SARSA** = Q-Learning with `Q[next_state, next_action]` instead of `max`. **Dyna-Q** = Q-Learning + model + planning. **IQL** = Q-Learning x agents. **DQN** = Q-Learning with a network. **PPO** = Actor-Critic + old log probs + clipped ratio.